<a href="https://colab.research.google.com/github/mahdisahad/WOA-Feature-Selection/blob/main/notebooks/06_Results_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="text-align: right">

# نوت‌بوک ۶: ارزیابی نهایی و تحلیل نتایج

این نوت‌بوک ویژگی‌هایی را که الگوریتم نهنگ در نوت‌بوک ۰۵ انتخاب کرد، روی **داده‌ی test** ارزیابی می‌کند و با مدل «همه‌ی ویژگی‌ها» مقایسه می‌کند.

**طراحی مقایسه:**
- دو مدل کاملاً یکسان (استانداردسازی + Logistic Regression با همان تنظیمات) آموزش می‌بینند؛ تنها تفاوت **مجموعه‌ی ویژگی‌ها** است: یکی همه‌ی ویژگی‌ها، دیگری فقط ویژگی‌های انتخاب‌شده‌ی WOA.
- هر دو فقط روی train آموزش می‌بینند و روی test (که در انتخاب ویژگی نقشی نداشت) ارزیابی می‌شوند، پس مقایسه منصفانه است.
- معیارها: Accuracy، Balanced Accuracy، Precision، Recall و F1 (Macro).

**نکته‌ها برای تفسیر:**
- test نسبتاً کوچک است؛ اختلاف یکی دو نمونه در accuracy ممکن است صرفاً نوسان تصادفی باشد.
- مدل پایه در این نوت‌بوک Logistic Regression است. مدل‌های دیگر (مثل SVM) در نوت‌بوک ۰۳ ارزیابی شده‌اند و ممکن است روی همه‌ی ویژگی‌ها بهتر باشند.
- هدف انتخاب ویژگی معمولاً **کوچک‌تر کردن مدل با حفظ کیفیت** است، نه لزوماً بالا بردن دقت.

</div>

<div dir="rtl" style="text-align: right">

### اتصال به Google Drive
Drive وصل می‌شود تا نتایج نوت‌بوک ۰۵ و دیتاست‌ها در دسترس باشند.

</div>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

<div dir="rtl" style="text-align: right">

### وارد کردن کتابخانه‌ها
- `numpy`، `pandas` و `matplotlib` برای محاسبه، جدول و نمودار.
- `Pipeline`، `StandardScaler` و `LogisticRegression` برای ساخت دو مدل.
- از `sklearn.metrics` معیارهای ارزیابی (`accuracy_score`، `balanced_accuracy_score`، `precision_score`، `recall_score`، `f1_score`)، `classification_report`، `confusion_matrix` و `ConfusionMatrixDisplay` برای نمایش ماتریس درهم‌ریختگی.

</div>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

print("Libraries imported successfully.")

<div dir="rtl" style="text-align: right">

### بازسازی دقیق داده‌ی train و test
همان مراحل نوت‌بوک‌های قبلی تکرار می‌شود تا **دقیقاً همان** train و test و همان ستون‌ها به دست آید: بارگذاری، تقسیم (با `random_state=42`)، حذف ویژگی‌های تکراری و ثابت از روی train. این مهم است چون نام ویژگی‌های انتخاب‌شده باید در همین ستون‌ها وجود داشته باشد. `class_labels` فهرست مرتب‌شده‌ی کلاس‌هاست که در گزارش‌ها و ماتریس‌ها استفاده می‌شود (برای دیتاست دوکلاسه یا چندکلاسه مثل TCGA کار می‌کند).

</div>

In [ ]:
PROJECT_PATH = Path("/content/drive/MyDrive/WOA_Project")

import sys
sys.path.append(str(PROJECT_PATH))
from dataset_config import *   # <-- dataset-specific code lives only here

RESULTS_PATH = get_results_path(PROJECT_PATH)

df = load_dataset(PROJECT_PATH)

X, y, groups = split_xy(df)

X_train, X_test, y_train, y_test, groups_train = split_train_test(X, y, groups)

duplicate_columns = X_train.columns[
    X_train.T.duplicated()
].tolist()

X_train = X_train.drop(columns=duplicate_columns)
X_test = X_test.drop(columns=duplicate_columns)
X_train, X_test = drop_near_constant(X_train, X_test)

class_labels = sorted(y_train.unique())

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

<div dir="rtl" style="text-align: right">

### خواندن ویژگی‌های انتخاب‌شده‌ی WOA
فایل `final_woa_selected_features.csv` (خروجی نوت‌بوک ۰۵) از پوشه‌ی نتایج دیتاست فعال خوانده می‌شود و نام ویژگی‌ها به یک فهرست تبدیل می‌شود. تعداد و بیست نام اول چاپ می‌شود. اگر این فایل نباشد یعنی نوت‌بوک ۰۵ برای این دیتاست اجرا نشده است.

</div>

In [ ]:
selected_features_df = pd.read_csv(
    RESULTS_PATH / "final_woa_selected_features.csv"
)

selected_features = selected_features_df["Feature"].tolist()

print("Number of selected features:", len(selected_features))
print("First 20 selected features:")
print(selected_features[:20])

<div dir="rtl" style="text-align: right">

### اعتبارسنجی فهرست ویژگی‌ها
سه شرط بررسی می‌شود: فهرست خالی نباشد، ویژگی تکراری نداشته باشد، و همه‌ی ویژگی‌ها در ستون‌های `X_train` وجود داشته باشند. اگر شرطی برقرار نباشد، برنامه با خطا متوقف می‌شود تا با داده‌ی نامنطبق ادامه ندهیم (مثلاً اگر فایل نتیجه مال دیتاست دیگری باشد).

</div>

In [ ]:
assert len(selected_features) > 0
assert len(set(selected_features)) == len(selected_features)
assert set(selected_features).issubset(set(X_train.columns))

print("Selected features validation passed.")

<div dir="rtl" style="text-align: right">

### ساخت دو نسخه‌ی داده
- **Baseline:** همه‌ی ویژگی‌ها (`X_train` و `X_test` بدون تغییر).
- **WOA:** فقط ستون‌های انتخاب‌شده (`X_train[selected_features]`).

`.copy()` نسخه‌ی مستقل می‌سازد. ابعاد چاپ می‌شود؛ تعداد نمونه‌ها یکسان و تعداد ستون‌ها متفاوت است.

</div>

In [ ]:
X_train_baseline = X_train.copy()
X_test_baseline = X_test.copy()

X_train_woa = X_train[selected_features].copy()
X_test_woa = X_test[selected_features].copy()

print("Baseline training shape:", X_train_baseline.shape)
print("WOA training shape:", X_train_woa.shape)

print("Baseline testing shape:", X_test_baseline.shape)
print("WOA testing shape:", X_test_woa.shape)

<div dir="rtl" style="text-align: right">

### تعریف دو مدل یکسان
دو `Pipeline` با تنظیمات یکسان (استانداردسازی و Logistic Regression با `class_weight="balanced"`، `max_iter=5000`، `random_state=42`) ساخته می‌شود. چون مدل‌ها یکسان‌اند، هر تفاوت در نتیجه فقط از مجموعه‌ی ویژگی‌ها می‌آید.

</div>

In [ ]:
baseline_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=5000,
        random_state=42
    ))
])

woa_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=5000,
        random_state=42
    ))
])

print("Both models are ready.")

<div dir="rtl" style="text-align: right">

### آموزش مدل‌ها
هر مدل با `fit` روی train خودش آموزش می‌بیند: مدل پایه با همه‌ی ویژگی‌ها و مدل WOA با ویژگی‌های انتخاب‌شده. استانداردسازی هم در همین مرحله فقط از train یاد گرفته می‌شود.

</div>

In [ ]:
baseline_model.fit(X_train_baseline, y_train)

woa_model.fit(X_train_woa, y_train)

print("Both models trained successfully.")

<div dir="rtl" style="text-align: right">

### پیش‌بینی روی test
هر مدل برچسب نمونه‌های test را پیش‌بینی می‌کند. این‌جا اولین بار است که test توسط مدل دیده می‌شود. پیش‌بینی‌ها (`y_pred_baseline` و `y_pred_woa`) چاپ می‌شوند.

</div>

In [ ]:
y_pred_baseline = baseline_model.predict(X_test_baseline)

y_pred_woa = woa_model.predict(X_test_woa)

print("Predictions completed.")
print("Baseline predictions:", y_pred_baseline)
print("WOA predictions:", y_pred_woa)

<div dir="rtl" style="text-align: right">

### محاسبه‌ی معیارها و جدول مقایسه
تابع `evaluate_model` پنج معیار را حساب می‌کند:
- **Accuracy:** درصد پیش‌بینی‌های درست.
- **Balanced Accuracy:** میانگین recall هر کلاس.
- **Precision / Recall / F1 (Macro):** برای هر کلاس جدا حساب و با وزن برابر میانگین‌گیری می‌شود، پس کلاس کوچک هم به اندازه‌ی کلاس بزرگ اثر دارد.

نتیجه‌ی دو مدل در یک جدول (`comparison_df`) کنار هم قرار می‌گیرد و اسم سطرها تعداد ویژگی‌ها را نشان می‌دهد. این جدول اصلی‌ترین نتیجه‌ی پروژه است.

</div>

In [ ]:
def evaluate_model(y_true, y_pred):

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced Accuracy": balanced_accuracy_score(y_true, y_pred),
        "Precision Macro": precision_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "Recall Macro": recall_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "F1 Macro": f1_score(
            y_true, y_pred, average="macro", zero_division=0
        )
    }


baseline_metrics = evaluate_model(
    y_test,
    y_pred_baseline
)

woa_metrics = evaluate_model(
    y_test,
    y_pred_woa
)

comparison_df = pd.DataFrame(
    [baseline_metrics, woa_metrics],
    index=[f"Baseline ({X_train.shape[1]} Features)", f"WOA ({len(selected_features)} Features)"]
)

comparison_df

<div dir="rtl" style="text-align: right">

### گزارش طبقه‌بندی مدل پایه
برای هر کلاس Precision (از پیش‌بینی‌های این کلاس چند درصد درست بود)، Recall (از نمونه‌های واقعی این کلاس چند درصد پیدا شد)، F1 و تعداد نمونه‌ی واقعی (support) نمایش داده می‌شود. برای دیدن این‌که مدل روی کدام کلاس ضعیف است مفید است.

</div>

In [ ]:
print("Baseline Classification Report")

print(
    classification_report(
        y_test,
        y_pred_baseline,
        labels=class_labels,
        zero_division=0
    )
)

<div dir="rtl" style="text-align: right">

### گزارش طبقه‌بندی مدل WOA
همان گزارش برای مدل با ویژگی‌های انتخاب‌شده. مقایسه‌ی آن با گزارش قبل نشان می‌دهد کاهش ویژگی‌ها روی کدام کلاس‌ها اثر مثبت یا منفی داشته است.

</div>

In [ ]:
print("WOA Classification Report")

print(
    classification_report(
        y_test,
        y_pred_woa,
        labels=class_labels,
        zero_division=0
    )
)

<div dir="rtl" style="text-align: right">

### ماتریس درهم‌ریختگی مدل پایه
ردیف‌ها کلاس واقعی و ستون‌ها کلاس پیش‌بینی‌شده‌اند. قطر اصلی پیش‌بینی‌های درست و خانه‌های بیرون قطر خطاها هستند؛ می‌بینیم کدام کلاس‌ها با هم اشتباه می‌شوند.

</div>

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_baseline,
    labels=class_labels,
    display_labels=[f"Class {c}" for c in class_labels],
    cmap="Blues",
    ax=ax
)

ax.set_title("Baseline Confusion Matrix")
plt.show()

<div dir="rtl" style="text-align: right">

### ماتریس درهم‌ریختگی مدل WOA
همان ماتریس برای مدل WOA. مقایسه‌ی دو ماتریس تعداد نمونه‌هایی را که با کاهش ویژگی‌ها درست یا غلط شده‌اند نشان می‌دهد.

</div>

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_woa,
    labels=class_labels,
    display_labels=[f"Class {c}" for c in class_labels],
    cmap="Blues",
    ax=ax
)

ax.set_title("WOA Confusion Matrix")
plt.show()

<div dir="rtl" style="text-align: right">

### نمودارهای مقایسه
دو نمودار کنار هم:
- **چپ:** مقایسه‌ی F1 Macro دو مدل (محور عمودی ۰ تا ۱).
- **راست:** تعداد ویژگی‌های هر مدل.

تصویری که از نمودارها معمولاً می‌خواهیم نشان دهیم: کیفیت تقریباً برابر با ویژگی‌های بسیار کمتر.

</div>

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Compare F1 Macro
comparison_df["F1 Macro"].plot(
    kind="bar",
    ax=axes[0],
    title="F1 Macro Comparison",
    ylim=(0, 1)
)

axes[0].set_ylabel("F1 Macro")
axes[0].tick_params(axis="x", rotation=15)

# Compare feature counts
axes[1].bar(
    ["Baseline", "WOA"],
    [X_train_baseline.shape[1], X_train_woa.shape[1]]
)

axes[1].set_title("Number of Selected Features")
axes[1].set_ylabel("Number of Features")

plt.tight_layout()
plt.show()

<div dir="rtl" style="text-align: right">

### درصد کاهش ویژگی‌ها
$$\text{reduction} = \frac{n_{all} - n_{selected}}{n_{all}} \times 100$$

تعداد ویژگی‌های اولیه، انتخاب‌شده و درصد کاهش چاپ می‌شود.

</div>

In [ ]:
original_feature_count = X_train_baseline.shape[1]
selected_feature_count = X_train_woa.shape[1]

reduction_percentage = (
    (original_feature_count - selected_feature_count)
    / original_feature_count
) * 100

print("Original features:", original_feature_count)
print("Selected features:", selected_feature_count)
print(f"Feature reduction: {reduction_percentage:.2f}%")

<div dir="rtl" style="text-align: right">

### ذخیره‌ی نتایج نهایی
در `results/<نام دیتاست>/` ذخیره می‌شود:
- `final_model_comparison.csv`: جدول مقایسه‌ی دو مدل،
- `baseline_classification_report.csv` و `woa_classification_report.csv`: گزارش کامل هر کلاس برای هر مدل (به‌صورت دیکشنری که به جدول تبدیل شده).

این فایل‌ها برای گزارش و ارائه استفاده می‌شوند.

</div>

In [ ]:
comparison_df.to_csv(
    RESULTS_PATH / "final_model_comparison.csv"
)

baseline_report = classification_report(
    y_test,
    y_pred_baseline,
    labels=class_labels,
    output_dict=True,
    zero_division=0
)

woa_report = classification_report(
    y_test,
    y_pred_woa,
    labels=class_labels,
    output_dict=True,
    zero_division=0
)

pd.DataFrame(baseline_report).to_csv(
    RESULTS_PATH / "baseline_classification_report.csv"
)

pd.DataFrame(woa_report).to_csv(
    RESULTS_PATH / "woa_classification_report.csv"
)

print("Final evaluation results saved successfully.")

<div dir="rtl" style="text-align: right">

### گزارش کامل PDF
این سلول همه‌ی نتایج را در یک فایل PDF **انگلیسی** (برای جلوگیری از مشکل فونت) جمع می‌کند: ابعاد و آمار داده و نمودارهای آن، پیش‌پردازش، مدل‌های پایه، اجرای WOA، مقایسه‌ی نهایی، گزارش هر کلاس، ماتریس‌های درهم‌ریختگی و فهرست ویژگی‌های انتخاب‌شده. از متغیرهای سلول‌های بالا و فایل‌های CSV پوشه‌ی نتایج استفاده می‌کند، پس باید بعد از اجرای همه‌ی سلول‌های قبلی اجرا شود. خروجی: `results/<نام دیتاست>/<نام دیتاست>_full_report.pdf`

</div>

In [ ]:
# Full PDF results report: dataset overview, preprocessing, baselines, WOA, final comparison.
# English only (no font problems). Uses variables from the cells above + CSV files in RESULTS_PATH.
from matplotlib.backends.backend_pdf import PdfPages
from sklearn.decomposition import PCA
from datetime import datetime

PDF_PATH = RESULTS_PATH / f"{ACTIVE_DATASET}_full_report.pdf"
PAGE = (11.69, 8.27)                      # A4 landscape
C_BASE, C_WOA = "#4C78A8", "#F58518"      # baseline = blue, WOA = orange
TAB10 = plt.get_cmap("tab10")
page_no = [0]

# ---------------------------------------------------------------- helpers
def fmt(df):
    out = df.copy()
    for c in out.columns:
        if pd.api.types.is_float_dtype(out[c]):
            v = out[c].dropna()
            whole = len(v) > 0 and (v == v.round()).all()
            out[c] = out[c].map(lambda x: "" if pd.isna(x) else (f"{x:.0f}" if whole else f"{x:.4f}"))
    return out.astype(str)

def new_page(title):
    fig = plt.figure(figsize=PAGE)
    fig.text(0.04, 0.965, title, fontsize=17, fontweight="bold", va="top")
    fig.text(0.96, 0.965, f"Dataset: {ACTIVE_DATASET}", fontsize=10, color="#666666", va="top", ha="right")
    fig.add_artist(plt.Line2D([0.04, 0.96], [0.915, 0.915], color="#cccccc", lw=0.8))
    page_no[0] += 1
    fig.text(0.96, 0.02, f"Page {page_no[0]}", ha="right", fontsize=8, color="#888888")
    return fig

def block(fig, rect, title):
    """Axes at rect=[left, bottom, width, height] with a small bold title above it."""
    ax = fig.add_axes(rect)
    fig.text(rect[0], rect[1] + rect[3] + 0.012, title, fontsize=11, fontweight="bold", va="bottom")
    return ax

def style(ax):
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    ax.grid(axis="y", color="#e5e5e5", lw=0.8); ax.set_axisbelow(True)

def draw_table(ax, df, fontsize=9, scale=1.45, index=True):
    """Table filling the block width; the index is drawn as the first column."""
    ax.axis("off")
    d = fmt(df)
    if index:
        d.insert(0, "", [str(i) for i in df.index])
    cols = [str(c) for c in d.columns]
    lens = np.array([max([len(cols[j])] + [len(str(v)) for v in d.iloc[:, j]]) for j in range(len(cols))], float) + 3
    widths = (lens / lens.sum()).tolist()
    ax_h_in = ax.get_position().height * PAGE[1]
    h = min(1.0, (len(d) + 1) * 0.30 * scale / ax_h_in / 1.45)
    tab = ax.table(cellText=d.values, colLabels=cols, colWidths=widths, cellLoc="center",
                   bbox=[0, 1 - h, 1, h])
    tab.auto_set_font_size(False); tab.set_fontsize(fontsize)
    for (r, c), cell in tab.get_celld().items():
        cell.set_edgecolor("#BBBBBB")
        if r == 0:
            cell.set_facecolor("#E8EEF4"); cell.set_text_props(weight="bold")
        elif index and c == 0:
            cell.set_facecolor("#F6F6F6"); cell.set_text_props(weight="bold")

def not_found(ax, name):
    ax.axis("off")
    ax.text(0, 0.9, f"{name} not found (run the earlier notebooks first)", color="#b00020", fontsize=9, va="top")

def grouped_bars(ax, df, cols, colors=None, ylim=(0, 1.25), labels=True):
    """df index = groups on the x axis, cols = one bar per column."""
    n = len(cols); w = 0.8 / n; x = np.arange(len(df.index))
    for j, c in enumerate(cols):
        bars = ax.bar(x + (j - (n - 1) / 2) * w, df[c].values, w, label=str(c),
                      color=(colors[j] if colors else TAB10(j)))
        if labels:
            for b in bars:
                ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.01, f"{b.get_height():.3f}",
                        ha="center", va="bottom", fontsize=6.5)
    ax.set_xticks(x); ax.set_xticklabels([str(i) for i in df.index], fontsize=8)
    ax.set_ylim(*ylim); ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
    ax.legend(fontsize=8, frameon=False, ncol=min(n, 3), loc="upper center"); style(ax)

def read_result(name, index_col=None):
    p = RESULTS_PATH / name
    return pd.read_csv(p, index_col=index_col) if p.exists() else None

def heatmap(ax, cm, title, fmt_str="d", cmap="Blues"):
    im = ax.imshow(cm, cmap=cmap, vmin=0, vmax=max(cm.max(), 1e-9))
    ax.set_xticks(range(len(class_names))); ax.set_xticklabels(class_names, fontsize=8)
    ax.set_yticks(range(len(class_names))); ax.set_yticklabels(class_names, fontsize=8)
    ax.set_xlabel("Predicted", fontsize=8); ax.set_ylabel("True", fontsize=8)
    ax.set_title(title, fontsize=10)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, format(cm[i, j], fmt_str), ha="center", va="center", fontsize=9,
                    color="white" if cm[i, j] > cm.max() / 2 else "black")

# ---------------------------------------------------------------- numbers used below
class_names = [str(c) for c in class_labels]
n_samples, n_feat_raw = X.shape
n_train, n_test = len(X_train), len(X_test)
n_dup = len(duplicate_columns)
n_feat_clean = X_train.shape[1]
n_const = n_feat_raw - n_dup - n_feat_clean
n_sel = len(selected_features)
reduction = (n_feat_clean - n_sel) / n_feat_clean * 100
counts_all = y.value_counts().reindex(class_labels).fillna(0).astype(int)
counts_tr = y_train.value_counts().reindex(class_labels).fillna(0).astype(int)
counts_te = y_test.value_counts().reindex(class_labels).fillna(0).astype(int)

scaler_r = StandardScaler().fit(X_train)
Z = scaler_r.transform(X_train)

with PdfPages(PDF_PATH) as pdf:

    # ============================================================ cover
    fig = plt.figure(figsize=PAGE); page_no[0] += 1
    fig.text(0.06, 0.88, "WOA Feature Selection", fontsize=30, fontweight="bold", va="top")
    fig.text(0.06, 0.79, "Full Results Report", fontsize=20, va="top", color="#444444")
    fig.text(0.06, 0.71, f"Dataset: {ACTIVE_DATASET}", fontsize=14, va="top")
    fig.text(0.06, 0.67, f"Generated: {datetime.now():%Y-%m-%d %H:%M}", fontsize=10, va="top", color="#666666")
    ax = block(fig, [0.06, 0.34, 0.50, 0.20], "Key Results (Test Set)")
    key = comparison_df[["Accuracy", "Balanced Accuracy", "F1 Macro"]].copy()
    key.insert(0, "Features", [n_feat_clean, n_sel])
    draw_table(ax, key, fontsize=10, scale=1.7)
    fig.text(0.66, 0.60, "Contents", fontsize=13, fontweight="bold", va="top")
    for i, t in enumerate(["1. Dataset Overview", "2. Feature Statistics", "3. Preprocessing",
                           "4. Baseline Models", "5. WOA Optimisation", "6. Final Comparison",
                           "7. Per-Class Results", "8. Confusion Matrices", "9. Selected Features"]):
        fig.text(0.66, 0.56 - i * 0.045, t, fontsize=11, va="top")
    pdf.savefig(fig); plt.close(fig)

    # ============================================================ 1. dataset overview
    fig = new_page("1. Dataset Overview")
    ax = block(fig, [0.04, 0.12, 0.40, 0.70], "Dataset Dimensions")
    dims = [("Samples", n_samples), ("Features (raw)", n_feat_raw), ("Classes", len(class_labels)),
            ("Class labels", ", ".join(class_names)),
            ("Subjects (groups)", int(groups.nunique()) if groups is not None else "-"),
            ("Missing values (features)", int(X.isnull().sum().sum())),
            ("Missing values (target)", int(y.isnull().sum())),
            ("Train samples", n_train), ("Test samples", n_test),
            ("Duplicate features removed", n_dup), ("Constant features removed", n_const),
            ("Features after cleaning", n_feat_clean), ("Features selected by WOA", n_sel)]
    draw_table(ax, pd.DataFrame(dims, columns=["Item", "Value"]).set_index("Item"), fontsize=10, scale=1.6)
    ax = block(fig, [0.55, 0.50, 0.41, 0.32], "Class Distribution (All Samples)")
    bars = ax.bar(class_names, counts_all.values, color=[TAB10(i) for i in range(len(class_names))])
    for b in bars:
        ax.text(b.get_x() + b.get_width() / 2, b.get_height(), f"{int(b.get_height())}", ha="center", va="bottom", fontsize=8)
    ax.set_ylabel("Samples"); style(ax)
    ax = block(fig, [0.55, 0.12, 0.41, 0.28], "Class Counts")
    ct = pd.DataFrame({"All": counts_all, "Percent": [f"{v:.2f}" for v in counts_all / counts_all.sum() * 100],
                       "Train": counts_tr, "Test": counts_te})
    ct.index = class_names
    draw_table(ax, ct, fontsize=9, scale=1.4)
    pdf.savefig(fig); plt.close(fig)

    # ============================================================ 2. feature statistics
    fm, fs, fmin, fmax = X.mean(), X.std(), X.min(), X.max()
    fig = new_page("2. Feature Statistics")
    ax = block(fig, [0.04, 0.60, 0.55, 0.24], "Per-Feature Statistics (summary across all features)")
    q = [0, .25, .5, .75, 1]
    summ = pd.DataFrame({"Min": [s.quantile(0) for s in (fm, fs, fmin, fmax)],
                         "25%": [s.quantile(.25) for s in (fm, fs, fmin, fmax)],
                         "Median": [s.quantile(.5) for s in (fm, fs, fmin, fmax)],
                         "75%": [s.quantile(.75) for s in (fm, fs, fmin, fmax)],
                         "Max": [s.quantile(1) for s in (fm, fs, fmin, fmax)]},
                        index=["Feature mean", "Feature std", "Feature min", "Feature max"])
    summ = summ.map(lambda v: f"{v:.4g}") if hasattr(summ, "map") else summ.applymap(lambda v: f"{v:.4g}")
    draw_table(ax, summ, fontsize=9, scale=1.5)
    ax = block(fig, [0.66, 0.60, 0.30, 0.24], "Overall Scale")
    overall = pd.DataFrame({"Value": [f"{fm.mean():.4g}", f"{fs.mean():.4g}", f"{fmin.min():.4g}", f"{fmax.max():.4g}"]},
                           index=["Mean of feature means", "Mean of feature std", "Global minimum", "Global maximum"])
    draw_table(ax, overall, fontsize=9, scale=1.5)
    ax = block(fig, [0.06, 0.10, 0.40, 0.38], "Distribution of Feature Means")
    ax.hist(np.sign(fm) * np.log10(1 + fm.abs()), bins=40, color=C_BASE)
    ax.set_xlabel("sign(mean) x log10(1 + |mean|)"); ax.set_ylabel("Features"); style(ax)
    ax = block(fig, [0.55, 0.10, 0.40, 0.38], "Distribution of Feature Std (log10)")
    pos = fs[fs > 0]
    ax.hist(np.log10(pos), bins=40, color=C_WOA); ax.set_xlabel("log10(feature std)"); ax.set_ylabel("Features"); style(ax)
    pdf.savefig(fig); plt.close(fig)

    # ---- sample feature histograms (same idea as notebook 01)
    fig = new_page("2. Feature Statistics - Sample Features")
    idx = np.linspace(0, X.shape[1] - 1, 6).astype(int)
    for k, j in enumerate(idx):
        r, c = divmod(k, 3)
        ax = block(fig, [0.06 + c * 0.31, 0.50 - r * 0.42, 0.25, 0.30], f"Feature: {str(X.columns[j])[:28]}")
        ax.hist(X.iloc[:, j].dropna(), bins=30, color=C_BASE); ax.set_ylabel("Samples"); style(ax)
    pdf.savefig(fig); plt.close(fig)

    # ---- PCA projection
    n_comp = min(10, Z.shape[0] - 1, Z.shape[1])
    pca = PCA(n_components=n_comp, svd_solver="randomized", random_state=42)
    P = pca.fit_transform(Z)
    fig = new_page("2. Feature Statistics - PCA of Training Samples")
    ax = block(fig, [0.06, 0.12, 0.50, 0.70], "PCA Projection (first 2 components)")
    ytr = y_train.to_numpy()
    for i, cl in enumerate(class_labels):
        m = ytr == cl
        ax.scatter(P[m, 0], P[m, 1], s=14, alpha=0.75, color=TAB10(i), label=f"Class {cl}")
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0] * 100:.1f}%)")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1] * 100:.1f}%)")
    ax.legend(fontsize=8, frameon=False); style(ax)
    ax = block(fig, [0.65, 0.12, 0.30, 0.70], "Explained Variance Ratio")
    ax.bar(range(1, n_comp + 1), pca.explained_variance_ratio_ * 100, color=C_BASE)
    ax.set_xlabel("Component"); ax.set_ylabel("Explained variance (%)"); style(ax)
    pdf.savefig(fig); plt.close(fig)

    # ============================================================ 3. preprocessing
    fig = new_page("3. Preprocessing")
    ax = block(fig, [0.04, 0.50, 0.42, 0.32], "Train / Test Split")
    split = pd.DataFrame({"Samples": [n_train, n_test],
                          "Share (%)": [f"{n_train / n_samples * 100:.1f}", f"{n_test / n_samples * 100:.1f}"]},
                         index=["Train", "Test"])
    draw_table(ax, split, fontsize=10, scale=1.6)
    ax = block(fig, [0.04, 0.10, 0.42, 0.28], "Standardised Training Data (check)")
    chk = pd.DataFrame({"Value": [f"{Z.mean(axis=0).mean():.4g}", f"{Z.std(axis=0).mean():.4g}",
                                  f"{Z.min():.4g}", f"{Z.max():.4g}"]},
                       index=["Mean of feature means", "Mean of feature std", "Minimum", "Maximum"])
    draw_table(ax, chk, fontsize=10, scale=1.5)
    ax = block(fig, [0.60, 0.15, 0.36, 0.62], "Number of Features at Each Stage")
    stages = ["Raw", "After duplicate\nremoval", "After constant\nremoval", "Selected\nby WOA"]
    vals = [n_feat_raw, n_feat_raw - n_dup, n_feat_clean, n_sel]
    bars = ax.barh(stages[::-1], vals[::-1], color=[C_WOA, C_BASE, C_BASE, C_BASE])
    for b in bars:
        ax.text(b.get_width(), b.get_y() + b.get_height() / 2, f" {int(b.get_width())}", va="center", fontsize=9)
    ax.set_xlim(0, max(vals) * 1.15); ax.set_xlabel("Features"); style(ax)
    pdf.savefig(fig); plt.close(fig)

    # ============================================================ 4. baseline models
    cv_df = read_result("baseline_cv_results.csv", 0)
    te_df = read_result("baseline_test_results.csv", 0)
    fig = new_page("4. Baseline Models (All Features)")
    ax = block(fig, [0.04, 0.60, 0.34, 0.22], "Cross-Validation Results (Train)")
    draw_table(ax, cv_df, fontsize=9, scale=1.5) if cv_df is not None else not_found(ax, "baseline_cv_results.csv")
    ax = block(fig, [0.42, 0.60, 0.54, 0.22], "Test Set Results")
    draw_table(ax, te_df, fontsize=9, scale=1.5) if te_df is not None else not_found(ax, "baseline_test_results.csv")
    ax = block(fig, [0.06, 0.10, 0.40, 0.38], "Cross-Validation Scores")
    if cv_df is not None: grouped_bars(ax, cv_df, list(cv_df.columns))
    ax = block(fig, [0.55, 0.10, 0.40, 0.38], "Test Scores")
    if te_df is not None: grouped_bars(ax, te_df, ["Accuracy", "Balanced Accuracy", "F1 Macro"])
    pdf.savefig(fig); plt.close(fig)

    # ============================================================ 5. WOA optimisation
    conv = read_result("woa_convergence.csv")
    runs = read_result("woa_multiple_runs_summary.csv")
    st = WOA_SETTINGS[ACTIVE_DATASET]
    fig = new_page("5. WOA Optimisation")
    ax = block(fig, [0.06, 0.52, 0.50, 0.30], "Convergence Curve")
    if conv is not None:
        ax.plot(conv["Iteration"], conv["Best_Fitness"], marker="o", color=C_WOA)
        ax.set_xlabel("Iteration"); ax.set_ylabel("Best fitness"); style(ax)
    else:
        not_found(ax, "woa_convergence.csv")
    ax = block(fig, [0.66, 0.52, 0.30, 0.30], "WOA Parameters")
    par = pd.DataFrame({"Value": [st["population_size"], st["max_iterations"],
                                  ", ".join(map(str, st["random_seeds"])), 0.9, "-6 to 6",
                                  "Logistic Regression", f"{CV_FOLDS}-fold"]},
                       index=["Population size", "Iterations", "Seeds", "Alpha", "Position bounds", "Evaluator", "CV"])
    draw_table(ax, par, fontsize=9, scale=1.4)
    ax = block(fig, [0.06, 0.08, 0.40, 0.28], "WOA Runs by Seed")
    draw_table(ax, runs.set_index(runs.columns[0]), fontsize=9, scale=1.5) if runs is not None else not_found(ax, "woa_multiple_runs_summary.csv")
    ax = block(fig, [0.60, 0.08, 0.36, 0.28], "Selected Features per Seed")
    if runs is not None:
        best_i = int(runs["Best Fitness"].idxmin())
        cols = [C_WOA if i == best_i else C_BASE for i in runs.index]
        bars = ax.bar(runs["Seed"].astype(str), runs["Selected Features"], color=cols)
        for b in bars:
            ax.text(b.get_x() + b.get_width() / 2, b.get_height(), f"{int(b.get_height())}", ha="center", va="bottom", fontsize=8)
        ax.set_xlabel("Seed (best run in orange)"); ax.set_ylabel("Features"); style(ax)
    pdf.savefig(fig); plt.close(fig)

    # ============================================================ 6. final comparison
    fig = new_page("6. Final Comparison: All Features vs WOA")
    ax = block(fig, [0.04, 0.62, 0.92, 0.20], "Test Set Metrics")
    draw_table(ax, comparison_df, fontsize=10, scale=1.7)
    ax = block(fig, [0.06, 0.10, 0.55, 0.40], "Metric Comparison")
    cmp_t = comparison_df.T
    cmp_t.columns = ["Baseline (all features)", "WOA (selected features)"]
    grouped_bars(ax, cmp_t, list(cmp_t.columns), colors=[C_BASE, C_WOA])
    ax = block(fig, [0.72, 0.10, 0.24, 0.40], "Number of Features")
    bars = ax.bar(["Baseline", "WOA"], [n_feat_clean, n_sel], color=[C_BASE, C_WOA])
    for b in bars:
        ax.text(b.get_x() + b.get_width() / 2, b.get_height(), f"{int(b.get_height())}", ha="center", va="bottom", fontsize=9)
    ax.set_xlabel(f"Reduction: {reduction:.2f}%", fontsize=9); style(ax)
    pdf.savefig(fig); plt.close(fig)

    # ============================================================ 7. per-class results
    rb = pd.DataFrame(baseline_report).T.astype(float)
    rw = pd.DataFrame(woa_report).T.astype(float)
    for _r in (rb, rw):
        _r.loc["accuracy", "support"] = _r.loc["macro avg", "support"]   # total number of test samples
    fig = new_page("7. Per-Class Results")
    ax = block(fig, [0.04, 0.52, 0.44, 0.32], "Classification Report - Baseline")
    draw_table(ax, rb, fontsize=8, scale=1.3)
    ax = block(fig, [0.52, 0.52, 0.44, 0.32], "Classification Report - WOA")
    draw_table(ax, rw, fontsize=8, scale=1.3)
    ax = block(fig, [0.06, 0.08, 0.88, 0.30], "F1 Score per Class")
    f1 = pd.DataFrame({"Baseline (all features)": [rb.loc[c, "f1-score"] for c in class_names],
                       "WOA (selected features)": [rw.loc[c, "f1-score"] for c in class_names]},
                      index=[f"Class {c}" for c in class_names])
    grouped_bars(ax, f1, list(f1.columns), colors=[C_BASE, C_WOA])
    pdf.savefig(fig); plt.close(fig)

    # ============================================================ 8. confusion matrices
    cm_b = confusion_matrix(y_test, y_pred_baseline, labels=class_labels)
    cm_w = confusion_matrix(y_test, y_pred_woa, labels=class_labels)
    nrm = lambda m: m / np.maximum(m.sum(axis=1, keepdims=True), 1)
    fig = new_page("8. Confusion Matrices")
    heatmap(block(fig, [0.07, 0.50, 0.36, 0.34], "Baseline (counts)"), cm_b, "")
    heatmap(block(fig, [0.55, 0.50, 0.36, 0.34], "WOA (counts)"), cm_w, "")
    heatmap(block(fig, [0.07, 0.07, 0.36, 0.34], "Baseline (row-normalised)"), nrm(cm_b), "", ".2f")
    heatmap(block(fig, [0.55, 0.07, 0.36, 0.34], "WOA (row-normalised)"), nrm(cm_w), "", ".2f")
    pdf.savefig(fig); plt.close(fig)

    # ============================================================ 9. selected features
    per_col, n_cols = 52, 4
    per_page = per_col * n_cols
    for start in range(0, max(n_sel, 1), per_page):
        chunk = selected_features[start:start + per_page]
        fig = new_page(f"9. Selected Features ({n_sel} total, {start + 1}-{start + len(chunk)})")
        for c in range(n_cols):
            part = [str(s)[:30] for s in chunk[c * per_col:(c + 1) * per_col]]
            fig.text(0.05 + c * 0.235, 0.89, "\n".join(part), fontsize=7.5, va="top", family="monospace")
        pdf.savefig(fig); plt.close(fig)

print("Full report saved at:", PDF_PATH)
